# QLoRA fine-tuning for code refinement — Colab runner (clean, single-pass)

Runs the full `coderefine` pipeline end to end: clone → install → train → evaluate →
forgetting check → report → package results for download. Every step shells out to
the same `coderefine` CLI used locally, so results are reproducible outside Colab too.

**Before running: Runtime → Change runtime type → T4 GPU (or better).**

Then **Runtime → Run all**. No manual intervention needed.

### What this notebook fixes vs. the original `colab_train.ipynb`
The response-template used for completion-only loss masking was being derived from a
synthetic probe (`"[/INST] AAAA"`) whose trailing space tokenizes differently than it
does in real examples, which all start with a code fence (`` ```py ``). SentencePiece
merges that trailing space into the fence token in real data, so the old template's
token-ID sequence never actually matched — the collator silently found the response
key in 0% of real examples, meaning training had no loss signal. This has been fixed
directly in `src/coderefine/train.py` (`find_response_template` now strips trailing
whitespace, and a loud coverage check aborts the run if match rate ever drops below
98% again). This notebook trains against that fixed code.

## 1 — GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 — Clone the repo

Curated `data/processed` and `data/benchmark` are committed (a few MB), so no need
for the raw 6 GB `Code_Refinement/*.jsonl` dumps.

In [ ]:
%cd /content
!rm -rf loRA-code-refinement
!git clone -q https://github.com/hamnaraeel/loRA-code-refinement.git
%cd loRA-code-refinement
!ls -la data/processed data/benchmark

## 2.5 — Mount Google Drive (checkpoint safety net)

Training runs on this config can take hours, and Colab's VM disk is wiped on
disconnect — including any checkpoints that were never copied out. Mounting
Drive here means the background sync started in section 5.5 has somewhere
durable to write to. If you'd rather not use Drive, skip this cell and section
5.5 below; everything else in the notebook works the same either way, you'll
just lose in-progress checkpoints on a disconnect (recoverable via the Files
panel as a manual, one-off download instead — slower and easy to forget).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/coderefine_runs"
import os
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print("Backing up to:", DRIVE_BACKUP_DIR)

## 3 — Install

Pinned to the exact combo verified end-to-end for this project: `transformers==4.46.3`
/ `peft==0.13.2` / `trl==0.11.4` / `accelerate==1.2.1`. Recent TRL releases replaced the
completion-only-loss collator with a chat-template-marker mechanism that TRL only
auto-patches for a small model allowlist (Mistral isn't on it) — installing a newer TRL
here would fail outright rather than silently mis-mask. `bitsandbytes` is left as a loose
`>=0.43.1` bound: exact-pinning it to older releases has been seen to break against this
transformers version (`triton.ops` import error) on current Colab images, so the newest
compatible build is safest even though its per-step 4-bit throughput on a T4 can vary
by allocation — see the note in the training cell below.

In [ ]:
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" \
    "bitsandbytes>=0.43.1" "datasets>=2.19" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib wandb
!pip install -q -e . --no-deps

import torch, transformers, peft, trl, accelerate, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| accelerate", accelerate.__version__)
print("bitsandbytes", bitsandbytes.__version__)

from trl import DataCollatorForCompletionOnlyLM  # must succeed — confirms the legacy masking path is active
print("legacy masking path: OK")

## 4 — Credentials (optional)

* **Hugging Face** — only needed for gated bases (e.g. Llama 3). Mistral-7B-Instruct-v0.3 is ungated.
* **Weights & Biases** — optional. Without a key the pipeline still logs everything to
  `artifacts/runs/<name>/metrics.jsonl`; it degrades, it does not fail.

In [ ]:
import os, getpass

# from huggingface_hub import login; login()   # only needed for gated models

use_wandb = False  # set True to log to W&B
if use_wandb:
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")
    os.environ["WANDB_PROJECT"] = "lora-code-refinement" 

## 5 — Sanity-check the masking fix before spending GPU time on it

Cheap (CPU, no model download): confirms the response-template token IDs actually
occur in the real training set before we pay for a multi-hour run on broken masking.

In [ ]:
import sys, json
sys.path.insert(0, "src")
from transformers import AutoTokenizer
from coderefine.train import find_response_template, render_dataset, _check_response_template_coverage

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
tok = AutoTokenizer.from_pretrained(BASE)
template = find_response_template(tok)
print("response_template:", repr(template))

rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
ds = render_dataset(rows, tok)
_check_response_template_coverage(tok, ds, template, sample_size=len(ds))
print("Masking coverage check passed — safe to train.")

## 5.5 — Start background checkpoint sync to Drive

Runs as a true background OS process (`&`, not an IPython background thread),
so it keeps syncing every 5 minutes even while the training cell below blocks
the notebook's kernel for hours. Skip this cell if you skipped the Drive mount
above.

In [ ]:
import subprocess

sync_script = f'''#!/bin/bash
while true; do
  rsync -a --exclude "*.tmp" artifacts/ "{DRIVE_BACKUP_DIR}/" 2>/dev/null
  sleep 300
done
'''
with open("/content/sync_loop.sh", "w") as f:
    f.write(sync_script)
subprocess.run(["chmod", "+x", "/content/sync_loop.sh"])
subprocess.Popen(
    ["/content/sync_loop.sh"],
    stdout=open("/content/sync_loop.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,  # survives the training cell blocking this kernel
)
print("Background Drive sync started — every 5 min, into", DRIVE_BACKUP_DIR)

## 6 — Train

One command, one config file. Everything that affects the result lives in the YAML.

**On runtime**: the README's ~50–70 min T4 estimate assumes a fast per-step rate.
On some Colab T4 allocations, 4-bit dequant throughput has been observed to run
several times slower depending on that session's specific GPU/driver pairing —
there is no reliable way to know in advance which you'll get. The cell below prints
timing for the first few steps; if `it/s` in the progress bar implies this run would
take much longer than you want to wait, interrupt this cell and rerun with the faster
line (1 epoch instead of 3, cutting total time ~3x) — swap which `!coderefine train`
line is commented out.

In [ ]:
# Full run (3 epochs, matches the paper numbers in the README):
!coderefine train configs/qlora_mistral7b.yaml

# Faster alternative if your T4 allocation is slow — uncomment to use instead:
# !coderefine train configs/qlora_mistral7b.yaml --set train.num_epochs=1

In [ ]:
# Loss curves from the local metric log (works with or without W&B)
import json, pathlib
import matplotlib.pyplot as plt

run = "qlora-mistral7b-r16"
rows = [json.loads(l) for l in (pathlib.Path("artifacts/runs")/run/"metrics.jsonl").read_text().splitlines() if l.strip()]
tr = [(r["_step"], r["loss"]) for r in rows if "loss" in r]
ev = [(r["_step"], r["eval_loss"]) for r in rows if "eval_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4.5))
if tr: ax.plot(*zip(*tr), label="train loss", lw=1.4)
if ev: ax.plot(*zip(*ev), label="eval loss", lw=1.4, marker="o")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title(run)
plt.tight_layout(); plt.show()

## 7 — Evaluate

Base first — it is the denominator of every improvement claim — then the tuned
model on the identical benchmark with the identical prompts and greedy decoding.

In [ ]:
ADAPTER = "artifacts/runs/qlora-mistral7b-r16/adapter"
BASE    = "mistralai/Mistral-7B-Instruct-v0.3"

!coderefine evaluate --split benchmark --base-model $BASE --tag base --load-in-4bit
!coderefine evaluate --split benchmark --base-model $BASE --adapter $ADAPTER --tag tuned --load-in-4bit
!coderefine compare artifacts/eval/base__benchmark.predictions.jsonl \
                    artifacts/eval/tuned__benchmark.predictions.jsonl

## 8 — Catastrophic forgetting check

In [ ]:
!coderefine forgetting --base-model $BASE --load-in-4bit
!coderefine forgetting --base-model $BASE --adapter $ADAPTER --load-in-4bit

## 9 — The sacred test split

Only run this once, after the configuration is frozen. Everything above used
validation and the benchmark.

In [ ]:
!coderefine evaluate --split test --final --base-model $BASE --tag base-test  --load-in-4bit
!coderefine evaluate --split test --final --base-model $BASE --adapter $ADAPTER --tag tuned-test --load-in-4bit
!coderefine compare artifacts/eval/base-test__test.predictions.jsonl \
                    artifacts/eval/tuned-test__test.predictions.jsonl \
                    --out-path artifacts/eval/comparison_test.json

## 10 — Report, package, and download

In [ ]:
!coderefine report
!coderefine export $ADAPTER --out-dir artifacts/release --base-model $BASE
from IPython.display import Markdown, display
display(Markdown(open("reports/EXPERIMENT_REPORT.md").read()))

In [ ]:
# Final sync (catches anything the last 5-min interval missed) and stop the background loop
import subprocess
subprocess.run(["rsync", "-a", "artifacts/", f"{DRIVE_BACKUP_DIR}/"])
subprocess.run(["pkill", "-f", "sync_loop.sh"])
print("Final sync to Drive done; background sync stopped.")

In [ ]:
# Zip the adapter + all evaluation artifacts and pull them down
!zip -qr coderefine_results.zip artifacts/release artifacts/eval artifacts/runs artifacts/forgetting reports data/processed/dataset_card.json data/benchmark/benchmark_card.json
!du -h coderefine_results.zip
from google.colab import files
files.download("coderefine_results.zip")

## 11 — Try the A/B server here (optional)

Serves base and fine-tuned from one resident model by toggling the adapter.

In [ ]:
import subprocess, time, requests, json
proc = subprocess.Popen(
    ["coderefine","serve","--base-model",BASE,"--adapter",ADAPTER,"--load-in-4bit","--port","8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/healthz", timeout=2).ok: break
    except Exception: time.sleep(2)
print(requests.get("http://127.0.0.1:8000/healthz").json())

payload = {
 "lang": "py",
 "old_code": "def load_config(path):\n    try:\n        with open(path) as fh:\n            return json.load(fh)\n    except Exception:\n        return None",
 "comment": "This except block swallows the error and returns None, which hides real failures. Just let it propagate.",
}
r = requests.post("http://127.0.0.1:8000/ab", json=payload, timeout=120)
print(json.dumps(r.json(), indent=2))